# Galea analysis

This notebook loads a session **with derived signals**, opens the signal browser, and adds analysis GUIs that run on the **browser's time range**.

* **Derived** channels are computed from the raw signals and appear in the browser group "Derived". They are described in the next section.
* The browser starts clipped to the session's **valid range**, if one was confirmed in `dataset_quality.ipynb`. Otherwise it shows the whole recording. To analyse a sub-range, zoom to it and press **clip to window**.
* **Pair analysis:** pick channels A and B, a preprocessing and an analysis (cross-correlation or coherence), each with its own options. **run** recomputes on the current range; **export figure** saves a PNG to `exports/`.

In [ ]:
%load_ext autoreload
%autoreload 2
from galea import load_session, resolve_data, valid_range
from galea.browser import SignalBrowser
from galea.pair_gui import PairGUI

DATA = None          # session folder path; None → folder dialog (or $GALEA_DATA if set)

## Derived signals
Each one is computed from raw signals when the session loads. NaN means not available: during dropouts, railing, or (for PPG-based signals) without stable skin contact.

**Muscle activity: `EMG<n> env`** (µV)
* For each EMG channel: band-pass 20–120 Hz with the 60 Hz notch, rectify, then low-pass 5 Hz.
* It tracks how strongly the muscle under the electrode contracts (jaw, brow, etc.). It is a relative measure: levels can't be compared across channels or sessions, because electrode contact differs.

**Head motion: `Linear accel |a|`, `Gyro |ω|`, `Pitch`, `Roll`** (from the IMU)
* An orientation filter (Madgwick) combines the gyroscope and accelerometer to estimate which way gravity points.
* **`Linear accel |a|`** (g): acceleration with gravity removed, so 0 at rest whatever the head's orientation. It shows translational movement and jolts.
* **`Gyro |ω|`** (deg/s): rotation speed of the head, in any direction.
* **`Pitch`** (deg, > 0 = nose up) and **`Roll`** (deg, > 0 = right ear down): head tilt relative to gravity. Turning the head left or right (yaw) can't be measured this way.
* **Calibration:** the IMU scales in the files are not physical, so each session is calibrated from its own data. The accelerometer is normalised so the reading at rest is 1 g; the gyro scale is fitted against the change in the gravity direction. The mounting axes (X forward, Y left, Z up) are assumed, not verified, so the signs of pitch and roll may be flipped.

**Heart: `Heart rate`** (bpm)
* Beats are detected on the PPG IR signal (the steepest point of each pulse upstroke) and converted to beat-to-beat rate. Implausible intervals, such as missed or extra beats, are dropped.
* Reliable where the PPG has stable skin contact; check it against `PPG IR` in the browser.

**Breathing: `Breathing (PPG)` and `Breathing phase`.** These are **indirect and approximate.**
* Nothing measures airflow or chest movement. The breathing signal is inferred from how breathing modulates the heart, in two ways:
  * heart rate rises during inspiration and falls during expiration (respiratory sinus arrhythmia);
  * pulse amplitude also varies with breathing.
* **`Breathing (PPG)`** (unitless, z-scored): both effects band-passed to 0.15–0.5 Hz (9–30 breaths/min), aligned in time and combined. Its peaks mark breathing cycles, but its amplitude is not breath depth.
* **`Breathing phase`** (deg): the cycle position, from the Hilbert transform. 0° is the peak of the breathing-linked heart rate, which physiologically falls **near the end of inspiration**; ±180° is near the end of expiration. This mapping is inferred from physiology, not measured, and can be off by a fraction of a cycle.
* **Limitations:**
  * It only works for breathing rates within 9–30 breaths/min.
  * It can't follow breaths shorter than about two heartbeats.
  * It weakens when the heart-rate modulation is small (e.g. older participants, fast or shallow breathing).
  * It is distorted by anything else that changes heart rate or pulse amplitude, such as movement, posture, or the PPG losing contact.
* Use it for breathing rate and approximate cycle timing. For accurate timing of inspiration and expiration, use a direct sensor (respiration belt, nasal thermistor or airflow).
* The breathing-run lines printed below report each run's rate and how well the heart-rate and pulse-amplitude estimates agree (|r| near 1 = consistent).

## Load

In [ ]:
s = load_session(resolve_data(DATA), derived=True)
vr = valid_range(s)
print(f"{s.name}: start {s.t0_wall():%Y-%m-%d %H:%M:%S %z}, duration {s.exg.t.iloc[-1]:.1f} s; valid range: "
      + (f"{vr[0]:.1f}–{vr[1]:.1f} s" if vr else "NOT CONFIRMED (whole recording; set it in dataset_quality.ipynb)"))
for r in s.resp_info:
    print(f"  breathing run {r['t0']:.1f}–{r['t1']:.1f} s: {r['beats']} beats, "
          f"{r['breaths_per_min']:.1f} breaths/min, HR–pulse-amplitude r = {r['hr_amp_r']:.2f}")

## Browser
It starts clipped to the valid range, if confirmed. Press **unclip** to see the whole recording.

In [ ]:
b = SignalBrowser(s, channels=["EEG7", "EOG0", "PPG IR", "Heart rate", "Breathing (PPG)", "Linear accel |a|"])
if vr:
    b.set_clip(*vr)
b

## Pair analysis

In [ ]:
pg = PairGUI(b, "Heart rate", "Breathing (PPG)", analysis="coherence")
pg